<a href="https://colab.research.google.com/github/tomo-mar/project_research_A/blob/main/Highlight_FewShot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1, モデルのロード & Driveマウント

In [ ]:
# 1. Driveマウントとディレクトリ設定
from google.colab import drive
import os

drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/VAD_Experiment"
INPUT_DIR = os.path.join(BASE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. ライブラリとモデルの準備
!pip install -q pydub librosa pandas numpy
!pip install -q git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview
!pip install -q qwen-omni-utils

import torch
import librosa
import re
import numpy as np
import pandas as pd
from pydub import AudioSegment
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor

import logging
logging.getLogger().setLevel(logging.ERROR)

model_path = "Qwen/Qwen2.5-Omni-7B"
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_path, torch_dtype=torch.bfloat16, device_map="auto", attn_implementation="sdpa",
)
processor = Qwen2_5OmniProcessor.from_pretrained(model_path)
print("\n初期設定とモデルロード完了")

2. few-shotデータ抽出関数の定義

In [ ]:
def prepare_full_rec_examples(input_dir, train_id="rec01"):
    """指定した録音データの「全チャンク」をお手本として抽出する"""
    print(f"{train_id} をFew-shotサンプルとして抽出中")
    examples = []

    csv_path = os.path.join(input_dir, f"{train_id}_label.csv")
    wav_path = os.path.join(input_dir, f"{train_id}.wav")

    if not (os.path.exists(csv_path) and os.path.exists(wav_path)):
        raise FileNotFoundError(f" {train_id} のファイル（wavまたはcsv）が見つからない")

    df = pd.read_csv(csv_path)
    audio = AudioSegment.from_wav(wav_path)
    chunk_length_ms = 5000

    for i, start_ms in enumerate(range(0, len(audio), chunk_length_ms)):
        end_ms = min(start_ms + chunk_length_ms, len(audio))
        chunk = audio[start_ms:end_ms]

        if len(chunk) < 1000:
            continue

        time_sec = start_ms / 1000.0

        # 相対時間が一致する正解ラベルの行を検索（誤差0.1秒許容）
        row = df[np.isclose(df['Relative_Time_sec'], time_sec, atol=0.1)]
        if row.empty:
            continue

        score = float(row['Label_norm'].iloc[0])

        # チャンクの保存
        save_path = f"/content/shot_{train_id}_{i}.wav"
        chunk.export(save_path, format="wav")

        examples.append({"path": save_path, "score": round(score, 2)})

    print(f" {len(examples)} 個のチャンクを抽出")
    return examples

3. プロンプトと推論関数の定義

In [ ]:
# プロンプトの定義
PROMPT_DIRECT = """
あなたは優秀な動画解析アシスタントです。
入力された5秒間の音声クリップを聞き、配信のハイライト（盛り上がり）確率を判定してください。

【評価基準】
0.0に近いほどベースライン（通常のトーン、特筆すべき起伏なし）
1.0に近いほどピーク（絶叫、爆笑など、最高潮の瞬間）

【出力形式】
判定結果となる「0.0 から 1.0 の間の数値（小数点以下2桁）」のみを出力してください。
"""

def run_qwen_few_shot(target_filepath, few_shot_examples):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are a highly capable audio analysis AI."}]}
    ]
    audio_arrays = []

    # 全チャンクをプロンプトの履歴に追加
    for shot in few_shot_examples:
        messages.append({
            "role": "user", "content": [
                {"type": "audio", "audio_url": shot["path"]},
                {"type": "text", "text": PROMPT_DIRECT}
            ]
        })
        messages.append({"role": "assistant", "content": [{"type": "text", "text": str(shot["score"])}]})
        arr, _ = librosa.load(shot["path"], sr=16000)
        audio_arrays.append(arr)

    # 推論したいターゲットを追加
    messages.append({
        "role": "user", "content": [
            {"type": "audio", "audio_url": target_filepath},
            {"type": "text", "text": PROMPT_DIRECT}
        ]
    })
    target_arr, _ = librosa.load(target_filepath, sr=16000)
    audio_arrays.append(target_arr)

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, audio=audio_arrays, return_tensors="pt", padding=True).to("cuda")

    generate_ids = model.generate(**inputs, max_new_tokens=20, return_audio=False)
    generate_ids = [cur_ids[len(inputs.input_ids[0]):] for cur_ids in generate_ids]
    output_text = processor.batch_decode(generate_ids, skip_special_tokens=True)[0]

    return output_text

def extract_probability(text):
    matches = re.findall(r"0\.\d+|1\.0+", text)
    if matches:
        return float(matches[-1])
    return 0.0

4. メイン処理

In [ ]:
target_id = "rec04"
target_wav = os.path.join(INPUT_DIR, f"{target_id}.wav")

# 1. ターゲットファイルの存在チェック
if not os.path.exists(target_wav):
    print(f"エラー: {target_wav} が見つからない")
else:
    # 2. 全データをFew-shotサンプルとして抽出
    few_shot_examples = prepare_full_rec_examples(INPUT_DIR, train_id="rec01")

    print(f"\n{target_id} のFew-shot推論を開始")
    audio = AudioSegment.from_wav(target_wav)
    results = []
    chunk_length_ms = 5000

    total_chunks = sum(1 for s in range(0, len(audio), chunk_length_ms) if len(audio[s:min(s + chunk_length_ms, len(audio))]) >= 1000)
    current_chunk = 0

    for start_ms in range(0, len(audio), chunk_length_ms):
        end_ms = min(start_ms + chunk_length_ms, len(audio))
        chunk = audio[start_ms:end_ms]

        if len(chunk) < 1000:
            continue

        current_chunk += 1
        time_sec = start_ms / 1000.0
        print(f"  [{current_chunk}/{total_chunks}]")

        temp_chunk_path = "/content/temp_chunk.wav"
        chunk.export(temp_chunk_path, format="wav")

        # 抽出した全サンプルを渡して推論
        res_text = run_qwen_few_shot(temp_chunk_path, few_shot_examples)
        score = extract_probability(res_text)

        results.append({
            "Time_sec": time_sec,
            "Score_Direct_FewShot": score,
            "RawText": res_text
        })

    # 3. CSV保存 (ファイル名を full_fewshot.csv にして区別)
    df = pd.DataFrame(results)
    output_csv_path = os.path.join(OUTPUT_DIR, f"{target_id}_full_fewshot.csv")
    df.to_csv(output_csv_path, index=False)
    print(f"\n結果を {output_csv_path} に保存完了")